# TFIM analysis

This notebook uses the shared scripts in `scripts/` and saves all paper figures and the SVM table under `Paper plots TFIM/`.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "data_config.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing scripts/data_config.py")

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "Paper Analysis" / "TFIM"
SCRIPTS_DIR = REPO_ROOT / "scripts"
PLOTS_DIR = NOTEBOOK_DIR / "Paper plots TFIM"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from plot_ipr_curves import plot_all_n_log_overlay, plot_ipr_curves
from plot_gram_matrices import plot_ipr_gram_matrix, plot_ipr_gram_matrices_up_to
from run_iprkernel_svm import run_iprkernel_svm_experiment, run_noisy_iprkernel_svm_ns_sweep

def save_pdf(fig, stem):
    path = PLOTS_DIR / f"{stem}.pdf"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

def clean_svm_results_frame(results):
    rows = []
    for i, L in enumerate(results["L"]):
        global_std = results["global_bal_acc_std"][i]
        local_std = results["local_bal_acc_std"][i]
        local_col = rf"Local bal. acc. ($\delta={results['local_delta']}$)"
        rows.append({
            "Experiment": "Clean",
            "$N_s$": "--",
            "$L$": int(L),
            "Global bal. acc.": rf"{results['global_bal_acc_mean'][i]:.4f} $\\pm$ {global_std:.4f}",
            local_col: rf"{results['local_bal_acc_mean'][i]:.4f} $\\pm$ {local_std:.4f}",
            "Splits": results["n_splits"],
            "Noise seeds": "--",
        })
    return pd.DataFrame(rows)


## IPR curves at L=10, raw scale

In [ ]:
fig, ax = plot_ipr_curves(
    dataset="tfim1d",
    L=10,
    curves=["IPR X", "IPR Z"],
    scale="log10",
    show_title=False,
)
save_pdf(fig, "tfim1d_L10_ipr_log10")

## Corresponding IPR Gram matrix

In [ ]:
fig, ax, gram = plot_ipr_gram_matrix(
    dataset="tfim1d",
    L=10,
    basis=("IPR X", "IPR Z"),
    normalize=True,
    show_title=False,
)
save_pdf(fig, "tfim1d_L10_ipr_gram_normalized")

## All-L IPR Gram matrices

In [ ]:
fig, axes, grams = plot_ipr_gram_matrices_up_to(
    dataset="tfim1d",
    L_list=list(range(10, 100, 10)),
    basis=("IPR X", "IPR Z"),
    normalize=False,
    ncols=3,
    show_title=False,
)
save_pdf(fig, "tfim1d_all_L_ipr_grams_raw_3x3")

In [ ]:
fig, axes, grams = plot_ipr_gram_matrices_up_to(
    dataset="tfim1d",
    L_list=list(range(10, 100, 10)),
    basis=("IPR X", "IPR Z"),
    normalize=True,
    ncols=3,
    show_title=False,
)
save_pdf(fig, "tfim1d_all_L_ipr_grams_normalized_3x3")

## All-L log IPR overlay

In [ ]:
fig, ax = plot_all_n_log_overlay(
    dataset="tfim1d",
    basis=("IPR X", "IPR Z"),
    mark_crossings=True,
    show_title=False,
)
save_pdf(fig, "tfim1d_all_L_ipr_XZ_log_crossings")

## TFIM IPR-kernel SVM performance

In [ ]:
fig, ax, svm_results = run_iprkernel_svm_experiment(
    dataset="tfim1d",
    L_values=list(range(10, 101, 10)),
    basis=("IPR X", "IPR Z"),
    phase_boundary=1.0,
    test_frac=0.10,
    local_delta=0.25,
    n_splits=100,
    rng_seed=12345,
    use_normalized_kernel=True,
    make_plot=True,
    show_plot_title=False,
    print_report=False,
)
ax.set_ylim(0.95, 1.001)
save_pdf(fig, "tfim1d_svm_performance_L10_to_L100_100splits")

## Noisy TFIM IPR-kernel SVM performance

In [ ]:
fig, ax, noisy_svm_table, noisy_svm_results = run_noisy_iprkernel_svm_ns_sweep(
    dataset="tfim1d",
    N_s_values=[200, 500, 1000, 2000, 5000, 10000],
    L_values=list(range(10, 101, 10)),
    basis=("IPR X", "IPR Z"),
    noise_seeds=list(range(10)),
    phase_boundary=1.0,
    test_frac=0.10,
    local_delta=0.25,
    n_splits=10,
    rng_seed=12345,
    use_normalized_kernel=True,
    make_plot=True,
    show_plot_title=False,
)
save_pdf(fig, "tfim1d_noisy_svm_performance_L10_to_L100_Ns_sweep_10splits_10seeds")

## SVM table

In [ ]:
clean_table = clean_svm_results_frame(svm_results).drop(columns=["Experiment", "$N_s$", "Noise seeds"])
clean_table_path = PLOTS_DIR / "tfim_standard_svm_results_latex.txt"
clean_table_path.write_text(clean_table.to_latex(index=False, escape=False), encoding="utf-8")

local_col = r"Local bal. acc. ($\delta=0.25$)"
noisy_table = pd.DataFrame({
    "$N_s$": noisy_svm_table["N_s"],
    "$L$": noisy_svm_table["L"],
    "Global bal. acc.": noisy_svm_table.apply(lambda row: rf"{row['global_mean']:.4f} $\\pm$ {row['global_std']:.4f}", axis=1),
    local_col: noisy_svm_table.apply(lambda row: rf"{row['local_mean']:.4f} $\\pm$ {row['local_std']:.4f}", axis=1),
    "Splits": noisy_svm_table["n_splits_per_noise_seed"],
    "Noise seeds": noisy_svm_table["n_noise_seeds"],
})
noisy_table_path = PLOTS_DIR / "tfim_noisy_svm_ns_sweep_results_latex.txt"
noisy_table_path.write_text(noisy_table.to_latex(index=False, escape=False), encoding="utf-8")

clean_table_path, noisy_table_path